# Integrated AI Practical Projects



---

## 1. Learning Objectives

- Build a complete end-to-end ML pipeline combining Days 1-10 skills.
- Apply cross-validation and GridSearchCV for hyperparameter tuning.
- Build an image preprocessing pipeline for OCR.
- Interpret ROC curves, AUC, and Precision-Recall curves.
- Create a model comparison dashboard tabulating multiple models.

## Concept / Theory

### The Complete ML Pipeline
```
Raw Data → EDA → Clean → Feature Engineering → Encode/Scale
    → Train/Validation/Test Split
    → Model Selection (cross-validation)
    → Hyperparameter Tuning (GridSearchCV / RandomSearchCV)
    → Final Evaluation on Test Set
    → Deploy
```

### Cross-Validation (CV)
Instead of a single train/test split, k-fold CV divides data into k folds, trains on k-1, tests on 1, and rotates. This gives a more reliable performance estimate.

### GridSearchCV
Systematically searches all combinations of hyperparameters using cross-validation. Expensive but thorough.

### RandomizedSearchCV
Samples a fixed number of random hyperparameter combinations. Faster than Grid when search space is large.

### Model Comparison
Always compare: LogReg → KNN → DT → RF → SVM → MLP
Start simple, add complexity only if needed.

In [ ]:
# Setup
!pip install -q scikit-learn pandas numpy matplotlib seaborn tensorflow

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings; warnings.filterwarnings('ignore')

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (train_test_split, cross_val_score,
                                      GridSearchCV, StratifiedKFold)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                               roc_curve, auc, precision_recall_curve,
                               average_precision_score, confusion_matrix,
                               classification_report, ConfusionMatrixDisplay)
import time
print('All libraries loaded.')

## Practical Lab 1 – Full End-to-End ML Pipeline

### Objective
Execute every step of the ML pipeline on the Breast Cancer dataset.

### Explanation
This lab integrates skills from Days 1-6: data loading, EDA, cleaning, feature engineering, modeling, and evaluation.

In [ ]:
# Lab 1: Load & EDA
cancer = load_breast_cancer()
X = pd.DataFrame(cancer.data, columns=cancer.feature_names)
y = pd.Series(cancer.target, name='target')

print('=== Step 1: Dataset Profile ===')
print(f'Shape: {X.shape}')
print(f'Missing values: {X.isnull().sum().sum()}')
print(f'Class distribution: 0 (malignant)={( y==0).sum()}, 1 (benign)={(y==1).sum()}')
print(f'\nBasic statistics:')
print(X.describe().round(2).iloc[:, :5])

In [ ]:
# Lab 1: EDA visualizations
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Class balance
axes[0].bar(['Malignant (0)', 'Benign (1)'], y.value_counts().sort_index(),
            color=['red', 'green'], alpha=0.8)
axes[0].set_title('Class Distribution'); axes[0].set_ylabel('Count')

# Feature distributions (first feature)
for label, color in [(0, 'red'), (1, 'green')]:
    axes[1].hist(X.iloc[y==label, 0], bins=20, color=color, alpha=0.6,
                 label=['Malignant','Benign'][label])
axes[1].set_title(cancer.feature_names[0]); axes[1].legend()

# Correlation heatmap (first 10 features)
corr = X.iloc[:, :10].corr()
sns.heatmap(corr, ax=axes[2], cmap='coolwarm', center=0, annot=False)
axes[2].set_title('Correlation (first 10 features)')
plt.tight_layout(); plt.show()

In [ ]:
# Lab 1: Prepare & train
X_tr, X_te, y_tr, y_te = train_test_split(
    X.values, y.values, test_size=0.2, stratify=y.values, random_state=42)

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  LogisticRegression(max_iter=500, random_state=42))
])
pipeline.fit(X_tr, y_tr)
y_pred  = pipeline.predict(X_te)
y_prob  = pipeline.predict_proba(X_te)[:, 1]

print('=== Step 5: Evaluation ===')
print(f'Accuracy:  {accuracy_score(y_te, y_pred):.4f}')
print(f'F1-Score:  {f1_score(y_te, y_pred):.4f}')
print(f'ROC-AUC:   {roc_auc_score(y_te, y_prob):.4f}')
print('\nClassification Report:')
print(classification_report(y_te, y_pred, target_names=['Malignant','Benign']))

## Practical Lab 2 – Cross-Validation & GridSearchCV

### Objective
Tune hyperparameters systematically without data leakage.

### Explanation
GridSearchCV finds the best parameters using cross-validation on the training set only. The test set remains untouched until final evaluation.

In [ ]:
# Lab 2: Cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(pipeline, X_tr, y_tr, cv=cv, scoring='roc_auc')
print('5-Fold Cross-Validation AUC:')
for i, s in enumerate(cv_scores, 1):
    print(f'  Fold {i}: {s:.4f}')
print(f'Mean: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

In [ ]:
# Lab 2: GridSearchCV
param_grid = {
    'model__C':        [0.01, 0.1, 1.0, 10.0],
    'model__max_iter': [200, 500],
    'model__solver':   ['lbfgs', 'saga']
}

grid = GridSearchCV(
    pipeline, param_grid, cv=5,
    scoring='roc_auc', n_jobs=-1, verbose=0
)
grid.fit(X_tr, y_tr)

print('Best parameters:', grid.best_params_)
print('Best CV AUC:', grid.best_score_.round(4))

best_model = grid.best_estimator_
best_pred  = best_model.predict(X_te)
best_prob  = best_model.predict_proba(X_te)[:, 1]
print(f'\nTest AUC after tuning: {roc_auc_score(y_te, best_prob):.4f}')

In [ ]:
# GridSearch results table
grid_df = pd.DataFrame(grid.cv_results_)
top_params = (grid_df[['param_model__C','param_model__solver','mean_test_score']]
              .sort_values('mean_test_score', ascending=False)
              .head(5).reset_index(drop=True))
print('Top 5 parameter combinations:')
print(top_params.to_string(index=False))

## Practical Lab 3 – Image Preprocessing Pipeline for OCR

### Objective
Build a reusable image preprocessing pipeline that improves OCR accuracy.

### Explanation
Good preprocessing (grayscale → denoise → threshold → deskew) can dramatically improve OCR accuracy on low-quality scans.

In [ ]:
# Lab 3: OCR preprocessing pipeline
import cv2
from PIL import Image, ImageDraw
import numpy as np

def create_noisy_doc():
    """Create a synthetic document with noise (simulates poor scan)."""
    img = Image.new('RGB', (400, 250), color=(245, 245, 240))  # off-white background
    draw = ImageDraw.Draw(img)
    draw.text((20, 20), 'Ministry of Transport', fill=(20, 20, 20))
    draw.text((20, 60), 'AI Training Certificate', fill=(20, 20, 20))
    draw.text((20, 100), 'Trainee: Aung Kyaw Zin', fill=(20, 20, 20))
    draw.text((20, 140), 'Score: 96 / 100', fill=(20, 20, 20))
    draw.text((20, 180), 'Date: 24 August 2026', fill=(20, 20, 20))
    img_arr = np.array(img)
    # Add salt-and-pepper noise
    noise = np.random.choice([0, 255], size=img_arr.shape[:2], p=[0.02, 0.98])
    img_arr[noise == 0] = 0
    return img_arr

orig = create_noisy_doc()

# Step 1: Grayscale
gray = cv2.cvtColor(orig, cv2.COLOR_RGB2GRAY)

# Step 2: Denoise
denoised = cv2.GaussianBlur(gray, (3,3), 0)

# Step 3: Adaptive threshold (binarize)
binary = cv2.adaptiveThreshold(denoised, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                cv2.THRESH_BINARY, 15, 5)

# Step 4: Deskew (rotate to correct small tilt)
def deskew(img):
    coords = np.column_stack(np.where(img < 200))
    if len(coords) < 10: return img
    angle = cv2.minAreaRect(coords)[-1]
    if angle < -45: angle += 90
    h, w = img.shape
    M = cv2.getRotationMatrix2D((w//2, h//2), angle, 1.0)
    return cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_CUBIC,
                           borderMode=cv2.BORDER_REPLICATE)
deskewed = deskew(binary)

# Visualize pipeline
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, img, title in zip(axes, [orig, gray, denoised, deskewed],
                           ['Original (noisy)', 'Grayscale', 'Denoised', 'Binarized+Deskew']):
    ax.imshow(img, cmap='gray' if img.ndim==2 else None)
    ax.set_title(title, fontsize=9); ax.axis('off')
plt.suptitle('OCR Preprocessing Pipeline')
plt.tight_layout(); plt.show()
print('Preprocessing pipeline complete!')

## 7. Practical Lab 4 – ROC, AUC & Precision-Recall Curves

### Objective
Visualize advanced evaluation metrics for binary classifiers.

### Explanation
- **ROC curve:** TP Rate vs FP Rate across all thresholds. AUC = area under this curve.
- **Precision-Recall curve:** Precision vs Recall. Better for imbalanced datasets.
- **Average Precision (AP):** Area under PR curve — summary metric.

In [ ]:
# Lab 4: Compare multiple classifiers on ROC
models_roc = {
    'Logistic Regression': best_model,
    'KNN':                 Pipeline([('sc', StandardScaler()), ('clf', KNeighborsClassifier(5))]),
    'Decision Tree':       Pipeline([('sc', StandardScaler()), ('clf', DecisionTreeClassifier(max_depth=5, random_state=42))]),
    'Random Forest':       Pipeline([('sc', StandardScaler()), ('clf', RandomForestClassifier(100, random_state=42))]),
}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for name, m in models_roc.items():
    if name != 'Logistic Regression':
        m.fit(X_tr, y_tr)
    probs = m.predict_proba(X_te)[:, 1]
    fpr, tpr, _ = roc_curve(y_te, probs)
    roc_auc_val = auc(fpr, tpr)
    axes[0].plot(fpr, tpr, lw=2, label=f'{name} (AUC={roc_auc_val:.3f})')
    prec, rec, _ = precision_recall_curve(y_te, probs)
    ap = average_precision_score(y_te, probs)
    axes[1].plot(rec, prec, lw=2, label=f'{name} (AP={ap:.3f})')

axes[0].plot([0,1],[0,1],'k--', label='Random')
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].set_title('ROC Curves'); axes[0].legend(loc='lower right', fontsize=8)
axes[0].grid(alpha=0.3)

axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curves'); axes[1].legend(loc='upper right', fontsize=8)
axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 8. Practical Lab 5 – Model Comparison Dashboard

### Objective
Build a comprehensive model comparison table.

### Explanation
A model comparison dashboard is the final step before model selection. It tabulates all models with their key metrics, training time, and prediction speed.

In [ ]:
# Lab 5: Full model comparison
all_models = {
    'Logistic Regression':    Pipeline([('sc', StandardScaler()), ('clf', LogisticRegression(C=1.0, max_iter=500, random_state=42))]),
    'KNN (k=5)':              Pipeline([('sc', StandardScaler()), ('clf', KNeighborsClassifier(5))]),
    'Decision Tree':          Pipeline([('sc', StandardScaler()), ('clf', DecisionTreeClassifier(max_depth=5, random_state=42))]),
    'Random Forest':          Pipeline([('sc', StandardScaler()), ('clf', RandomForestClassifier(100, random_state=42, n_jobs=-1))]),
    'Gradient Boosting':      Pipeline([('sc', StandardScaler()), ('clf', GradientBoostingClassifier(100, random_state=42))]),
    'SVM (RBF)':              Pipeline([('sc', StandardScaler()), ('clf', SVC(probability=True, random_state=42))]),
    'MLP':                    Pipeline([('sc', StandardScaler()), ('clf', MLPClassifier(hidden_layer_sizes=(64,32), max_iter=200, random_state=42))]),
}

results = []
for name, pipe in all_models.items():
    t0 = time.time(); pipe.fit(X_tr, y_tr); t_train = time.time()-t0
    t0 = time.time(); probs = pipe.predict_proba(X_te)[:,1]; t_pred = time.time()-t0
    preds = (probs >= 0.5).astype(int)
    results.append({
        'Model':     name,
        'Accuracy':  accuracy_score(y_te, preds),
        'F1':        f1_score(y_te, preds),
        'AUC':       roc_auc_score(y_te, probs),
        'Train_s':   round(t_train, 3),
        'Pred_ms':   round(t_pred*1000, 2)
    })

df_results = pd.DataFrame(results).sort_values('AUC', ascending=False).reset_index(drop=True)
df_results[['Accuracy','F1','AUC']] = df_results[['Accuracy','F1','AUC']].round(4)
print('=== Model Comparison Dashboard ===')
print(df_results.to_string(index=False))

# Visualize
fig, ax = plt.subplots(figsize=(10, 5))
df_results.plot(kind='bar', x='Model', y=['Accuracy','F1','AUC'], ax=ax,
                rot=30, alpha=0.85, edgecolor='white')
plt.title('Model Comparison — Breast Cancer Dataset')
plt.ylabel('Score'); plt.legend(loc='lower right')
plt.tight_layout(); plt.show()

## 9. Exercises

**Exercise 1:** Apply GridSearchCV to Random Forest on Breast Cancer. Try `n_estimators=[50,100,200]`, `max_depth=[None,5,10]`.

In [ ]:
# Exercise 1
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
rf_pipe = Pipeline([('sc', StandardScaler()), ('rf', RandomForestClassifier(random_state=42))])
param_grid = {'rf__n_estimators': [50,100,200], 'rf__max_depth': [None,5,10]}
# Run GridSearchCV

**Exercise 2:** Add Gradient Boosting and SVM to the model comparison. Which model wins?

In [ ]:
# Exercise 2 — Already included in Lab 5. Try on a different dataset (Wine or Digits)

**Exercise 3:** Implement RandomizedSearchCV (faster than Grid) on MLP. Explore different hidden layer sizes.

In [ ]:
# Exercise 3
from sklearn.model_selection import RandomizedSearchCV
from sklearn.neural_network import MLPClassifier
from scipy.stats import randint
mlp_pipe = Pipeline([('sc', StandardScaler()), ('mlp', MLPClassifier(max_iter=200, random_state=42))])
# Define param distributions and run RandomizedSearchCV

## 10. Challenge Task

Choose ANY dataset from `sklearn.datasets` that you haven't used before. Build a complete pipeline:
1. Load, profile, and clean
2. Feature engineering (at least one derived feature)
3. Compare at least 4 models using 5-fold CV
4. Tune the best model with GridSearchCV
5. Final evaluation on held-out test set
6. Present a model comparison dashboard
7. State your recommendation (which model would you deploy and why?)

In [ ]:
# Challenge Task — Your chosen dataset
from sklearn.datasets import load_wine  # or load_digits, fetch_california_housing, etc.
# Complete pipeline: load → EDA → clean → engineer → compare → tune → evaluate → report

## 11. Real-World AI/ML Application

### The AI/ML Development Lifecycle

```
1. Business Problem Definition
2. Data Collection & Labeling
3. EDA & Data Quality
4. Feature Engineering
5. Model Selection (CV)
   ↓
6. Hyperparameter Tuning (GridSearchCV)
7. Final Evaluation (test set)
8. Model Explainability
9. Deployment (API / Notebook / App)
10. Monitoring (drift detection)
```

Day 11 covers steps 4-7. Day 12 covers steps 8-9.

## 12. Day Summary

| Concept | Key Point |
|---|---|
| End-to-end pipeline | EDA → Clean → Model → Evaluate |
| 5-Fold CV | Reliable performance estimate without test data |
| GridSearchCV | Exhaustive hyperparameter search |
| ROC/AUC | Overall classification performance |
| Precision-Recall | Better metric for imbalanced data |
| Model dashboard | Compare Accuracy, F1, AUC, speed for all models |

**→ Day 12: Final Capstone Project**